# Inspect optimize-from-burnin results (CMA-ES trace)

Loads the results of `run_starships_optimize` / `submit_starships_optimize`
(see `starships.retrieval_optimize`): the maximum-a-posteriori search started
from a burn-in, using CMA-ES.

This notebook only reads the saved `.npz` files -- **no petitRADTRANS, no YAML
setup needed** (unlike `Analyse_retrieval_output_new.ipynb`), so it is fast to
open and re-run.

For each start, the `.npz` holds:
- `theta_best` / `lnprob_best`: the best point found and its lnprob (log-prior + log-likelihood).
- `theta_trace` / `lnprob_trace`: every candidate CMA-ES actually evaluated (not just
  the best one), in evaluation order -- see the module docstring of
  `starships/retrieval_optimize.py` for why this is kept.
- `history`: one row per CMA-ES generation (evaluations so far, best lnprob so far,
  generation median lnprob, step size).

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from starships.plotting_fcts import scatterplot_logl
from starships.retrieval_optimize import compare_results

%matplotlib inline

## 1. Load the results

In [ ]:
# --- Edit this: the .npz result files, one per CMA-ES start ---
WALKER_PATH = Path('/home/adb/scratch/DataAnalysis/SPIRou/walker_steps/KELT-20_b')
BURNIN_STEM = 'walker_steps_emission_HRR_WL_wfc3_g395H_1_g395H_2_spitzer_modif_disso_3355174_burnin'

result_files = [WALKER_PATH / f'optimum_{BURNIN_STEM}_start{i}.npz' for i in (1, 2, 3)]
for f in result_files:
    assert f.is_file(), f'missing: {f}'
print(f'{len(result_files)} result file(s) found.')

Same summary table as `run_starships_optimize --compare`: best `lnprob` per start,
number of evaluations, why each one stopped, and the best value found for every
parameter -- plus the spread between starts (a rough reliability check: starts that
land far apart from each other mean the optimizer has not converged, or the likelihood
is rugged/multimodal there).

In [ ]:
_ = compare_results(result_files)

## 2. logl vs each parameter (evaluation trace)

Every real `lnprob` call CMA-ES made, not just the best one -- same tool
(`scatterplot_logl`) as the MCMC posterior scatter in `Analyse_retrieval_output_new.ipynb`,
applied to the optimizer's trace instead of the sampler's chain. Near the optimum, the shape
of this scatter gives a rough, local (Laplace-type) picture of the posterior along each
parameter -- **not** the true posterior shape (it only reflects the neighbourhood of the
best point each start found, and misses skew or other modes; see the caveat in
`retrieval_optimize.py`'s module docstring).

**Coverage is not the same for every start** (a real incident, not by design): start 1 had
to restart from near scratch partway through, so its full trace up to convergence is saved.
Starts 2 and 3 ran for a long time before the trace feature existed, so only their last
portion of evaluations (from where they were `--resume`d onward) is in the trace -- the
cell below prints exactly how much for each.

In [ ]:
colors = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:red'}

param_names, fig, ax, is_first = None, None, None, True
for result_file, color in zip(result_files, colors.values()):
    d = np.load(result_file)
    if param_names is None:
        param_names = list(d['param_names'])

    n_trace, n_evals = len(d['lnprob_trace']), int(d['n_evals'])
    print(f"{result_file.stem}: {n_trace}/{n_evals} evaluations traced "
          f"({100 * n_trace / n_evals:.0f}%), lnprob range "
          f"[{d['lnprob_trace'].min():.2f}, {d['lnprob_trace'].max():.2f}]")

    fig, ax = scatterplot_logl(d['theta_trace'], d['lnprob_trace'], n_max_pts=10000,
                               tight_ylim=is_first, alpha=0.15, color=color,
                               ylabels=param_names, fig=fig, ax=ax)
    is_first = False

print('\ncolors: start 1 = blue (full trace), start 2 = orange, start 3 = red '
      '(both partial, see above)')

## Next: model generation

Not done in this notebook on purpose (kept small, optimizer-only). Each `theta_best`
loaded above is a plain `(n_dim,)` array in the same order as `retrieval.params_prior.keys()`
(`param_names` above) -- the same shape `Analyse_retrieval_output_new.ipynb`'s
"Choose which parameters you want" section expects for `best_params`, so it plugs
directly into `retrieval.unpack_theta(best_params)` and everything downstream
(spectrum, contributions, profiles) in that notebook, unchanged.